# Hands-on Agentic RAG

## Libraries and Dependencies

In [ ]:
!pip install llama-index openai
!pip install llama-index-embeddings-huggingface
!pip install llama-index-llms-huggingface
!pip install huggingface_hub[hf_xet]

In [ ]:
!pip install llama-index-postprocessor-cohere-rerank
!pip install llama-index-readers-file

In [ ]:
%pip install llama-index-vector-stores-qdrant
!pip install -U qdrant_client==1.14.1

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import StorageContext
from llama_index.llms.huggingface import HuggingFaceLLM
import openai
import os
import llama_index
import IPython
import qdrant_client
import torch

## Set up API Keys


In [ ]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
openai.api_key = OPENAI_API_KEY
COHERE_API_KEY = userdata.get("COHERE_API_KEY")

## Get and View Knowledge Base

A sample of the documents: https://jonfernandes.github.io/knowledge-base.html

In [ ]:
!mkdir data
!wget https://jonfernandes.github.io/html.zip -O "./data/html.zip"
!unzip -o data/html.zip -d data

In [ ]:
!ls data/html

In [ ]:
IPython.display.HTML("data/html/Are-there-any-wheelchair-friendly-taxis-in-London.html")

In [ ]:
IPython.display.HTML("data/html/Are-there-any-wheelchair-friendly-taxis-in-Paris.html")

## Documents

In [ ]:
documents = SimpleDirectoryReader("data", recursive=True).load_data()
documents

In [ ]:
len(documents)

In [ ]:
documents[0]

In [ ]:
print(documents[0].text)

In [ ]:
documents[1]

In [ ]:
print(documents[1].text)

In [ ]:
documents[0].metadata["file_path"]

In [ ]:
documents[1].metadata["file_path"]

## NAIVE RAG

In [ ]:
documents = SimpleDirectoryReader("data", recursive=True).load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()
response = query_engine.query("Where can I get help in London? I am at the station.")
print(response)

## Swapping Out Components

**Vector Database**

In [ ]:
documents = SimpleDirectoryReader("data", recursive=True).load_data()

client = qdrant_client.QdrantClient(":memory:")
client

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
OpenAIEmbedding()

In [ ]:
embed_model = OpenAIEmbedding(model='text-embedding-3-large')
embed_model

In [ ]:
os.environ['HUGGING_FACE_HUB_TOKEN'] = userdata.get("HUGGING_FACE_HUB_TOKEN")

https://huggingface.co/BAAI/bge-small-en-v1.5

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.embed_model = embed_model

In [ ]:
embedding = embed_model.get_text_embedding("Where can I get help in London? I am at the station.")

### Settings

Whenever a particular component is not provided, the Settings object is used to provide it as a global default.

In [ ]:
from llama_index.core import Settings
Settings

In [ ]:
from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(model="gpt-4o", temperature=0.0)
Settings

## Bi-Encoder

In [ ]:
vector_store = QdrantVectorStore(client=client, collection_name="knowledgebase")
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
)

query_engine = index.as_query_engine()
response = query_engine.query("Where can I get help in London? I am at the station.")
print(response)

In [ ]:
response.source_nodes

In [ ]:
len(response.source_nodes)

In [ ]:
print(response.source_nodes[0].metadata["file_path"])
IPython.display.HTML(response.source_nodes[0].text)

In [ ]:
print(response.source_nodes[1].metadata["file_path"])
IPython.display.HTML(response.source_nodes[1].text)

## Cross-Encoder (Reranking)

In [ ]:
from llama_index.postprocessor.cohere_rerank import CohereRerank

cohere_rerank = CohereRerank(api_key=COHERE_API_KEY, top_n=2)

In [ ]:
documents = SimpleDirectoryReader("data", recursive=True).load_data()
index = VectorStoreIndex.from_documents(documents, recursive=True)
query_engine = index.as_query_engine(similarity_top_k=5,
                                     node_postprocessors=[cohere_rerank])
response = query_engine.query("Where can I get help in London? I am at the station.")
print(response.response)

In [ ]:
print(response.source_nodes[0].metadata["file_path"])
IPython.display.HTML(response.source_nodes[0].text)

In [ ]:
print(response.source_nodes[1].metadata["file_path"])
IPython.display.HTML(response.source_nodes[1].text)

## Evaluating RAG

RAG pipelines need measurement to improve. Without evals you can't tell whether a change to your retriever, LLM, or chunking strategy actually made things better or worse.

We'll use **LlamaIndex's built-in evaluation module** (`llama_index.core.evaluation`) to measure two key quality signals against the Eurostar knowledge base:

| Metric | What it measures | Why it matters |
|--------|-----------------|----------------|
| **Faithfulness** | Is the answer grounded in the retrieved context? | Catches hallucinations |
| **Answer Relevancy** | Does the answer actually address the question asked? | Catches off-topic responses |

Both evaluators use the LLM configured in `Settings.llm` as the judge — no extra libraries required.

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.evaluation import DatasetGenerator

# Load the Eurostar knowledge base
eval_documents = SimpleDirectoryReader("data/html").load_data()

# Auto-generate evaluation questions from the knowledge base using the LLM
dataset_generator = DatasetGenerator.from_documents(
    eval_documents,
    num_questions_per_chunk=1,  # one question per document keeps the set manageable
    show_progress=True,
)

eval_questions = await dataset_generator.agenerate_questions_from_nodes()
print(f"Generated {len(eval_questions)} evaluation questions")

In [ ]:
for i, q in enumerate(eval_questions[:10]):
    print(f"{i+1}. {q}")

In [ ]:
from llama_index.core.evaluation import FaithfulnessEvaluator
from llama_index.core import VectorStoreIndex, Settings

# Build a fresh query engine over the Eurostar knowledge base
index = VectorStoreIndex.from_documents(eval_documents)
query_engine = index.as_query_engine()

faithfulness_evaluator = FaithfulnessEvaluator(llm=Settings.llm)

# Single-question walkthrough
sample_q = eval_questions[0]
sample_response = await query_engine.aquery(sample_q)

result = await faithfulness_evaluator.aevaluate_response(response=sample_response)

print(f"Question:  {sample_q}")
print(f"Response:  {sample_response}")
print(f"Faithful:  {result.passing}  (score: {result.score})")
print(f"Feedback:  {result.feedback}")

In [ ]:
from llama_index.core.evaluation import RelevancyEvaluator

relevancy_evaluator = RelevancyEvaluator(llm=Settings.llm)

result = await relevancy_evaluator.aevaluate_response(
    query=sample_q,
    response=sample_response,
)

print(f"Question:  {sample_q}")
print(f"Response:  {sample_response}")
print(f"Relevant:  {result.passing}  (score: {result.score})")
print(f"Feedback:  {result.feedback}")

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)

from llama_index.core.evaluation import BatchEvalRunner

runner = BatchEvalRunner(
    {"faithfulness": faithfulness_evaluator, "relevancy": relevancy_evaluator},
    workers=4,
)

# Evaluate across the first 10 questions (to keep API costs manageable)
eval_subset = eval_questions[:10]
eval_results = await runner.aevaluate_queries(
    query_engine,
    queries=eval_subset,
)

# Summary stats
faithfulness_scores = [r.score for r in eval_results["faithfulness"] if r.score is not None]
relevancy_scores    = [r.score for r in eval_results["relevancy"]    if r.score is not None]

print(f"Faithfulness     — mean: {sum(faithfulness_scores)/len(faithfulness_scores):.2f}")
print(f"Answer Relevancy — mean: {sum(relevancy_scores)/len(relevancy_scores):.2f}")

# Per-question breakdown
rows = []
for i, q in enumerate(eval_subset):
    rows.append({
        "Question": q[:70] + ("…" if len(q) > 70 else ""),
        "Faithfulness": eval_results["faithfulness"][i].score,
        "Relevancy": eval_results["relevancy"][i].score,
    })

pd.DataFrame(rows)

### Interpreting the Results

| Score range | Faithfulness | Answer Relevancy |
|-------------|-------------|-----------------|
| ≥ 0.8 | Answers are well-grounded — low hallucination risk | Answers address the question correctly |
| 0.5 – 0.8 | Some grounding issues; review retrieved context | Retrieval may not be surfacing the right documents |
| < 0.5 | High hallucination risk | Poor retrieval alignment |

**How to act on low scores:**

- **Low Faithfulness** → try a more capable LLM, tighten chunk size, or add explicit source-grounding instructions to your prompt
- **Low Answer Relevancy** → improve retrieval with better embeddings, reranking (Cohere), or a higher `similarity_top_k`

Evals give you a feedback loop: make a change, re-run, compare scores.

## Agentic RAG (post-live training exercise)

### async()

**What “async” is for**

Async lets your program start one task, then switch to another while the first one is waiting (usually on network or disk). This is great for things like LLM calls, API requests, database reads, etc.

**Event loop:** Think of it as a traffic controller. It decides which async task runs next.

One event loop per thread: In a normal Python script, you typically have one loop for the main thread.

asyncio.run(...): The usual way to start an async program in a script. It creates the event loop, runs your main async function, then shuts the loop down.

Call it once in a typical script.

**Notebooks are different**

In Jupyter/Colab, an event loop is often already running, so you usually don’t use asyncio.run().

You can often just use await directly.

**`async def' defines an async function (a coroutine).**

Calling it doesn’t run it immediately in the normal way—it returns something that needs the event loop to execute it.

**await means: “pause here until this finishes.”**

While paused, your code gives control back to the event loop so other tasks can run.

Key idea: async code makes progress by pausing at await points.

**Async = concurrency: tasks take turns.**

Only one task is running at any exact instant, but when one is waiting (e.g., for a network response), another can run.

This is why async is great for I/O-bound work (like LLM/API calls).

Not true parallelism: async doesn’t automatically use multiple CPU cores.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

def get_favourite_colour() -> str:
    '''Useful for getting a favourite colour'''
    return "Purple"

agent = FunctionAgent(
    llm = OpenAI(model="gpt-4o-mini"),
    tools = [get_favourite_colour],
    system_prompt = "You are a helpful assistant"
)

In [ ]:
response = await agent.run("What is my favourite colour")
print(response)

In [ ]:
response = await agent.run("Is this common choice for many people?")
print(response)

In [ ]:
from llama_index.core.workflow import Context

ctx = Context(agent)

response = await agent.run("What is my favourite colour", ctx=ctx)
print(response)

In [ ]:
response = await agent.run("Is this common choice for many people?", ctx=ctx)
print(response)

In [ ]:
documents = SimpleDirectoryReader("data", recursive=True).load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

async def get_context_knowledgebase(query):
    '''Useful for answering questions about railway travel'''
    response = await query_engine.aquery(query)
    return str(response)

In [ ]:
async def get_context_knowledgebase(query: str) -> str:
    response = await query_engine.aquery(query)
    return str(response)

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

def get_favourite_colour() -> str:
    '''Useful for getting a favourite colour'''
    return "Cyan"

agent = FunctionAgent(
    llm = OpenAI(model="gpt-4o-mini"),
    tools = [get_favourite_colour, get_context_knowledgebase],
    system_prompt = "You are a helpful assistant."
)

In [ ]:
response = await agent.run("Is this common for people", ctx=ctx)
print(response)

In [ ]:
response = await agent.run("Where can I get help in London? I am at the station.", ctx=ctx)
print(response)

Soli Deo Gloria